In [93]:
import pandas as pd
import numpy as np
import matplotlib as plts

In [94]:
weather_data = pd.read_csv("../data/raw/weather_data/thuringia_weather_2025.csv")
weather_data.head()


,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,shortwave_radiation,station_id,station_name
0,2025-01-01 00:00:00,-3.4,92,-4.4,0.0,982.7,99,10.3,224,0.0,377,Almerswind-Grümpen
1,2025-01-01 01:00:00,-2.6,93,-3.6,0.0,982.7,100,11.3,227,0.0,377,Almerswind-Grümpen
2,2025-01-01 02:00:00,-2.2,92,-3.3,0.0,982.4,100,12.6,227,0.0,377,Almerswind-Grümpen
3,2025-01-01 03:00:00,-2.2,93,-3.1,0.0,982.0,100,10.2,224,0.0,377,Almerswind-Grümpen
4,2025-01-01 04:00:00,-1.9,93,-3.0,0.0,981.6,100,11.2,221,0.0,377,Almerswind-Grümpen


In [ ]:
import pandas as pd

# Ensure time is datetime
weather_data['time'] = pd.to_datetime(weather_data['time'], errors='coerce')
weather_data = weather_data.dropna(subset=['time']).sort_values('time')

station_dfs = []

for station_name, group in weather_data.groupby('station_name'):
    # Set time as index
    group = group.set_index('time')
    
    # ***** CRITICAL: Remove duplicate timestamps within this station *****
    # Option 1: Keep the first occurrence (fast)
    group = group[~group.index.duplicated(keep='first')]
    
    # Option 2: Average duplicates (if you prefer)
    # group = group.groupby(group.index).mean()
    
    # Keep only numeric columns
    numeric_cols = group.select_dtypes(include=['number']).columns
    group_numeric = group[numeric_cols]
    
    # Resample to 15 minutes and interpolate
    group_15min = group_numeric.resample('15min').interpolate(method='linear')
    
    # Add station name back as a column
    group_15min['station_name'] = station_name
    station_dfs.append(group_15min.reset_index())

# Combine all stations
all_stations_15min = pd.concat(station_dfs, ignore_index=True)

Resampled shape: (5867211, 12)
                 time  temperature_2m  relative_humidity_2m  dew_point_2m  \
0 2025-01-01 00:00:00            -3.4                 92.00          -4.4   
1 2025-01-01 00:15:00            -3.2                 92.25          -4.2   
2 2025-01-01 00:30:00            -3.0                 92.50          -4.0   
3 2025-01-01 00:45:00            -2.8                 92.75          -3.8   
4 2025-01-01 01:00:00            -2.6                 93.00          -3.6   

   precipitation  surface_pressure  cloud_cover  wind_speed_10m  \
0            0.0             982.7        99.00           10.30   
1            0.0             982.7        99.25           10.55   
2            0.0             982.7        99.50           10.80   
3            0.0             982.7        99.75           11.05   
4            0.0             982.7       100.00           11.30   

   wind_direction_10m  shortwave_radiation  station_id        station_name  
0              224.00     

In [96]:
all_stations_15min

,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,shortwave_radiation,station_id,station_name
0,2025-01-01 00:00:00,-3.40,92.00,-4.40,0.0,982.700,99.00,10.30,224.00,0.0,377.0,Almerswind-Grümpen
1,2025-01-01 00:15:00,-3.20,92.25,-4.20,0.0,982.700,99.25,10.55,224.75,0.0,377.0,Almerswind-Grümpen
2,2025-01-01 00:30:00,-3.00,92.50,-4.00,0.0,982.700,99.50,10.80,225.50,0.0,377.0,Almerswind-Grümpen
3,2025-01-01 00:45:00,-2.80,92.75,-3.80,0.0,982.700,99.75,11.05,226.25,0.0,377.0,Almerswind-Grümpen
4,2025-01-01 01:00:00,-2.60,93.00,-3.60,0.0,982.700,100.00,11.30,227.00,0.0,377.0,Almerswind-Grümpen
...,...,...,...,...,...,...,...,...,...,...,...,...
5867206,2026-01-01 22:00:00,2.10,79.00,-1.10,0.1,979.500,100.00,32.60,242.00,0.0,240.0,Werningshausen 2
5867207,2026-01-01 22:15:00,2.05,79.75,-1.05,0.1,979.375,100.00,31.40,241.25,0.0,240.0,Werningshausen 2
5867208,2026-01-01 22:30:00,2.00,80.50,-1.00,0.1,979.250,100.00,30.20,240.50,0.0,240.0,Werningshausen 2
5867209,2026-01-01 22:45:00,1.95,81.25,-0.95,0.1,979.125,100.00,29.00,239.75,0.0,240.0,Werningshausen 2


In [100]:
all_stations_15min.to_csv("../data/raw/weather_data/15min_interpolated_weather.csv")